### 07. Understanding User Preferences and Preparing Interaction Data

**Objective:** Build a simulated user-item interaction dataset, convert it into a user-item matrix, run basic analysis, and explain how collaborative filtering uses these interactions to generate personalized recommendations.

**Input:** `../data/processed/ted_talks_cleaned.csv` (cleaned TED dataset)
**Output:** `../data/interaction/user_interactions.csv` (interaction data folder)

**⚠ Important:** The interaction data in this notebook is **simulated** for learning purposes. No real user data is used.

**Design:**
- **500 users × 200 TED Talks** (100,000 possible pairs), sparsely rated: ~9,800 interactions, so ~90% of the matrix is empty.
- Each user has 2–3 **preferred tags** (from the TED dataset) - they rate talks matching those tags higher.
- Rating scale: **1–5**.
- Additional signals: `liked`, `saved`, `watch_pct` (0–1), `timestamp`.

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

np.random.seed(42)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

##### Load Cleaned TED Dataset
We use the cleaned dataset so talks have normalized tags. We'll sample **200 talks** to keep the interaction data manageable.

In [13]:
df_talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
print(f"Full dataset: {df_talks.shape[0]} talks")

# Sample 200 talks (deterministic)
talk_pool = df_talks.sample(n=200, random_state=42).reset_index(drop=True)
talk_pool = talk_pool[["talk_id", "title", "tags"]].copy()

# Build a tag list per talk (lowercased)
def parse_tags(s):
    return [t.strip().lower() for t in str(s).split(",") if t.strip()]

talk_pool["tag_list"] = talk_pool["tags"].apply(parse_tags)

print(f"Sampled talks: {len(talk_pool)}")
talk_pool.head(3)

Full dataset: 4005 talks
Sampled talks: 200


,talk_id,title,tags,tag_list
0,995,glorious visions in animation and performance,"art, design, performance, technology, animation","[art, design, performance, technology, animation]"
1,1307,medicine for the 99 percent,"tedx, economics, global issues, health","[tedx, economics, global issues, health]"
2,1879,what is so special about the human brain,"brazil, brain, evolution, neuroscience","[brazil, brain, evolution, neuroscience]"


##### Define User Preferences
Each of the **500 users** is assigned **2–3 preferred tags**. When a user "sees" a talk, the probability they interact and the rating they give depends on how well the talk's tags match their preferences.

In [14]:
# All unique tags across the 200 sampled talks
all_tags = sorted({t for tags in talk_pool["tag_list"] for t in tags})
print(f"Unique tags in sampled talks: {len(all_tags)}")
print(all_tags[:20], "...")

N_USERS = 500
user_ids = [f"U{str(i).zfill(4)}" for i in range(1, N_USERS + 1)]

# Assign 2-3 preferred tags per user
user_prefs = {}
for uid in user_ids:
    k = np.random.choice([2, 3])
    user_prefs[uid] = set(np.random.choice(all_tags, size=k, replace=False).tolist())

# Quick sample
for uid in user_ids[:5]:
    print(f"{uid}: {sorted(user_prefs[uid])}")

Unique tags in sampled talks: 323
['activism', 'adventure', 'advertising', 'africa', 'aging', 'agriculture', 'ai', 'aids', 'algorithm', 'alternative energy', 'ancient world', 'animals', 'animation', 'anthropocene', 'archaeology', 'architecture', 'art', 'asteroid', 'astronomy', 'bacteria'] ...
U0001: ['farming', 'god']
U0002: ['computers', 'graphic design', 'social change']
U0003: ['google', 'intelligence', 'terrorism']
U0004: ['advertising', 'animation']
U0005: ['activism', 'egypt', 'religion']


##### Generate Simulated Interactions
For every (user, talk) pair we simulate:
- **Probability of interacting** (watching) rises with tag overlap.
- **Rating (1–5)** is higher when the talk matches the user's preferences.
- **liked** is True if rating ≥ 4.
- **saved** is True if rating = 5, or rating = 4 with 30% chance.
- **watch_pct** is drawn from a distribution shifted by rating.
- **timestamp** is random within the last 2 years.

In [15]:
rows = []
start_date = datetime(2023, 1, 1)
end_date   = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

for uid in user_ids:
    prefs = user_prefs[uid]
    for _, talk in talk_pool.iterrows():
        talk_tags = set(talk["tag_list"])
        overlap = len(prefs & talk_tags)

        # Probability of interacting (watching) the talk.
        # Base 8%, +30% per overlapping preferred tag. The base was calibrated
        # so that 500 users x 200 talks yields ~10,000 interactions (~90% sparse).
        p_watch = 0.08 + 0.30 * overlap
        if np.random.rand() > p_watch:
            continue

        # Rating: base 2.5 + 0.8 per overlap + noise
        rating_raw = 2.5 + 0.8 * overlap + np.random.normal(0, 0.7)
        rating = int(np.clip(round(rating_raw), 1, 5))

        liked = rating >= 4
        saved = (rating == 5) or (rating == 4 and np.random.rand() < 0.3)

        # watch_pct: correlated with rating
        watch_pct = float(np.clip(np.random.normal(loc=0.2 + rating * 0.15, scale=0.15), 0.0, 1.0))

        ts = start_date + timedelta(days=int(np.random.randint(0, date_range)))

        rows.append({
            "user_id": uid,
            "talk_id": int(talk["talk_id"]),
            "rating": rating,
            "liked": liked,
            "saved": saved,
            "watch_pct": round(watch_pct, 2),
            "timestamp": ts.strftime("%Y-%m-%d"),
        })

interactions = pd.DataFrame(rows)
print(f"Total interactions: {len(interactions)}")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique talks: {interactions['talk_id'].nunique()}")
print(f"Sparsity: {1 - len(interactions) / (N_USERS * len(talk_pool)):.2%} of the matrix is empty")
interactions.head()

Total interactions: 9778
Unique users: 500
Unique talks: 200
Sparsity: 90.22% of the matrix is empty


,user_id,talk_id,rating,liked,saved,watch_pct,timestamp
0,U0001,27155,3,False,False,0.99,2023-07-27
1,U0001,1375,3,False,False,0.70,2023-01-20
2,U0001,13029,4,True,False,1.00,2023-03-09
3,U0001,19266,2,False,False,0.42,2024-04-05
4,U0001,2722,2,False,False,0.43,2024-03-27


##### Save the Interaction Dataset

In [16]:
INTERACTION_DIR = Path("../data/interaction")
INTERACTION_DIR.mkdir(parents=True, exist_ok=True)

INTERACTION_PATH = INTERACTION_DIR / "user_interactions.csv"
interactions.to_csv(INTERACTION_PATH, index=False)
print(f"Saved: {INTERACTION_PATH}")

Saved: ..\data\interaction\user_interactions.csv


##### Build the User-Item Interaction Matrix
- **Rows:** users
- **Columns:** TED Talks
- **Values:** rating (1–5)
- **Missing values (NaN)** = the user has not interacted with that talk.

In [17]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
)

print(f"Matrix shape: {user_item_matrix.shape}  (users × talks)")
print(f"Non-null entries: {user_item_matrix.notna().sum().sum()}")
print(f"Missing entries: {user_item_matrix.isna().sum().sum()}")

# Preview the first 8x8 block
user_item_matrix.iloc[:8, :8]

Matrix shape: (500, 200)  (users × talks)
Non-null entries: 9778
Missing entries: 90222


talk_id,27,44,46,71,79,91,131,190
user_id,,,,,,,,
U0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0002,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0005,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
U0006,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
U0007,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
U0008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Basic Interaction Data Analysis

We answer the six questions asked in the task. Each is followed by a supporting code output.

In [18]:
# Q1 - Unique users
print(f"Q1. Unique users: {interactions['user_id'].nunique()}")

# Q2 - Unique talks
print(f"Q2. Unique TED Talks interacted with: {interactions['talk_id'].nunique()}")

Q1. Unique users: 500
Q2. Unique TED Talks interacted with: 200


In [19]:
# Q3 - TED Talks with the highest average rating (min 5 ratings to avoid small-sample bias)
top_rated = (
    interactions.groupby("talk_id")
    .agg(avg_rating=("rating", "mean"), n_ratings=("rating", "size"))
    .query("n_ratings >= 5")
    .sort_values("avg_rating", ascending=False)
    .head(10)
    .reset_index()
    .merge(talk_pool[["talk_id", "title"]], on="talk_id", how="left")
)
print("Q3. Top 10 highest-rated TED Talks (min 5 ratings):")
top_rated[["talk_id", "title", "avg_rating", "n_ratings"]]

Q3. Top 10 highest-rated TED Talks (min 5 ratings):


,talk_id,title,avg_rating,n_ratings
0,1534,a test for parkinson s with a phone call,3.018182,55
1,1562,the promise of research with stem cells,3.014706,68
2,29265,the biology of gender from dna to the brain,2.983607,61
3,372,design and the elastic mind,2.973684,38
4,2030,what s wrong with your pa w0rd,2.956522,46
5,566,a plug for smart power outlets,2.953488,43
6,9636,black life at the intersection of birth and death,2.937500,64
7,2543,why brexit happened and what to do next,2.924242,66
8,2589,a visual history of social dance in 25 moves,2.901408,71
9,55793,could a breathalyzer detect cancer,2.900000,60


> **Note on ratings:** Since interaction probability and rating both scale with tag overlap, most users give talks in their non-preferred topics low ratings (1-2). Average rating is ~2.7, with ~58% of interactions rated 3 or higher. In a real system, ratings would be more polarized (mostly 4-5 with a few 1s).

In [20]:
# Q4 - Users who rated the most talks
most_active_users = (
    interactions.groupby("user_id")
    .agg(n_interactions=("talk_id", "size"))
    .sort_values("n_interactions", ascending=False)
    .head(10)
)
print("Q4. Top 10 most-active users:")
most_active_users

Q4. Top 10 most-active users:


,n_interactions
user_id,
U0168,36
U0013,35
U0035,34
U0135,34
U0492,33
U0417,33
U0006,33
U0033,32
U0423,31


In [21]:
# Q5 - Talks with the most interactions
most_interacted_talks = (
    interactions.groupby("talk_id")
    .agg(n_interactions=("user_id", "size"), avg_rating=("rating", "mean"))
    .sort_values("n_interactions", ascending=False)
    .head(10)
    .reset_index()
    .merge(talk_pool[["talk_id", "title"]], on="talk_id", how="left")
)
print("Q5. Top 10 most-interacted TED Talks:")
most_interacted_talks[["talk_id", "title", "n_interactions", "avg_rating"]]

Q5. Top 10 most-interacted TED Talks:


,talk_id,title,n_interactions,avg_rating
0,2625,can a divided america heal,79,2.835443
1,1501,the game that can give you 10 extra years of life,73,2.780822
2,1826,how reliable is your memory,72,2.680556
3,2589,a visual history of social dance in 25 moves,71,2.901408
4,1993,comics that ask what if,69,2.739130
5,1562,the promise of research with stem cells,68,3.014706
6,2543,why brexit happened and what to do next,66,2.924242
7,9636,black life at the intersection of birth and death,64,2.937500
8,2265,the surprisingly logical minds of babies,63,2.793651
9,19740,how to build synthetic dna and send it across the internet,63,2.777778


##### 7. What Does a Missing Rating Mean?

A **missing value** in the user-item matrix means **the user has not interacted with that talk**. It does **not** mean they disliked it. Reasons could be:

1. **Not exposed** - the talk was never recommended or shown.
2. **No interest** - the user saw it but chose not to watch/rate.
3. **Watched but not rated** - implicit interaction without an explicit rating.
4. **Not released yet** at the time of interaction.

Collaborative filtering algorithms must distinguish **explicit missing** (never seen) from **implicit negative** (seen, disliked). Our simulated dataset only captures the *explicit* side - watched talks are rated.

##### 8. How Collaborative Filtering Uses This Data

**Core idea:** If two users rate several talks similarly, they likely share interests. A talk liked by User A but not yet seen by User B is a good candidate to recommend to User B.

**Example:**
- User A and User B both rated *"Averting the climate crisis"* 5, *"Simplicity sells"* 4, and *"The best stats you've ever seen"* 5.
- The system infers a high **user-user similarity** (cosine or Pearson).
- User A also gave 5 to *"The case for optimism on climate change"*, which User B hasn't interacted with.
- The system predicts User B would rate it highly -> **recommend it**.

**Two main approaches:**

| Approach | How it works | Needs |
|---|---|---|
| **User-based CF** | Find users similar to the target user; recommend talks they liked. | User-user similarity matrix. |
| **Item-based CF** | Find talks similar to ones the target user already liked; recommend those. | Item-item similarity matrix. |

**Why our interaction data enables this:**
- **User IDs** identify who is who.
- **Talk IDs** identify what was interacted with.
- **Ratings (1–5)** quantify preference strength.
- **Sparsity** (~90% empty) means most users only rate a small subset - exactly the realistic scenario CF is designed to handle.

**From the matrix:** rows are users, columns are talks, values are ratings. Once we have enough non-missing values, **CF can predict the missing ones** and rank talks per user.

##### Observations

1. **Simulated but realistic.** The dataset deliberately encodes tag-preference biases - users rate talks in their preferred topics higher. This creates a signal CF can learn.

2. **Sparse by design.** 90.2% of the 500 x 200 matrix is empty (9,778 of 100,000 cells filled). Each user rates ~20 of 200 talks (median 19, range 7-36); each talk receives ~49 ratings (range 33-79). This is the mid-density regime where collaborative filtering still has co-rating overlap to work with.

3. **Rating distribution is centred below 3.** The mean rating is ~2.7: 58% of interactions are rated 3+, 15% are rated 4+. Ratings are modest because the rating formula starts at 2.5 and only tag overlap pushes a talk higher.

4. **Active users vs. popular talks.** Interaction counts vary (7-36 ratings per user, 33-79 per talk), but the long tail is bounded by construction: every one of the 500 users and all 200 talks appear in the dataset, so there are no zero-interaction users or talks.

5. **Missing rating ≠ dislike.** A NaN means *no interaction*, not *negative rating*. Ignoring this distinction in CF leads to bad predictions.

6. **Extensible.** Adding more signals (likes, saves, watch_pct) allows the model to move from **explicit feedback** (ratings) to **implicit feedback** (behaviour), which typically improves recommendation quality in production.

7. **Next step (Task 8):** Use this matrix to compute **user-user similarity** and predict missing ratings. Then merge with the content-based recommender from Task 6 into a hybrid system.

In [22]:
# Final sanity checks
print("Final interaction data:")
print(f"  Rows: {len(interactions)}")
print(f"  Unique users: {interactions['user_id'].nunique()}")
print(f"  Unique talks: {interactions['talk_id'].nunique()}")
print(f"  Matrix shape: {user_item_matrix.shape}")
print(f"  Sparsity: {user_item_matrix.isna().mean().mean():.2%}")

ratings_per_user = interactions.groupby("user_id").size()
min_ratings_per_user = int(ratings_per_user.min())
print(f"  Min ratings per user: {min_ratings_per_user}")
print(f"  Median ratings per user: {ratings_per_user.median():.0f}")

expected_ids = {f"U{str(i).zfill(4)}" for i in range(1, N_USERS + 1)}
missing_ids = expected_ids - set(interactions["user_id"].unique())
print(f"  Missing user IDs: {len(missing_ids)}")

assert missing_ids == set(), "Every designed user must appear in the dataset"
assert min_ratings_per_user >= 3, "Every user must have at least 3 ratings"
print("  ✓ All 500 users are present and have at least 3 ratings")

Final interaction data:
  Rows: 9778
  Unique users: 500
  Unique talks: 200
  Matrix shape: (500, 200)
  Sparsity: 90.22%
  Min ratings per user: 7
  Median ratings per user: 19
  Missing user IDs: 0
  ✓ All 500 users are present and have at least 3 ratings


#### 📄 Task Report

The written report for this notebook lives at [`reports/task_07_observations.md`](../reports/task_07_observations.md).

- Markdown: [`reports/task_07_observations.md`](../reports/task_07_observations.md)
- PDF: [`reports/pdf/task_07_observations.pdf`](../reports/pdf/task_07_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
